# Lean-39 : Capstone — escalier vers la sous-série MIMO (détection par flips, budget de descente et digestion)

**Série** : [SymbolicAI/Lean](../README.md) — entrée/sortie de la sous-série [MIMO](MIMO/README.md).

**Carnets visés** : `MIMO-01-Descente-Budget-Lean-Python` → `MIMO-04-Capstone-Digestion-Python` (quatre carnets), avec leur lac compagnon [`mimo_lean/`](../mimo_lean/README.md) (13 fichiers `.lean`, 0 sorry, kernel `lean4-wsl`).

**Pourquoi un escalier** : la sous-série MIMO déplace 21c/21/21b dans un dossier `MIMO/` ([PR #20371](https://github.com/jsboige/CoursIA/pull/20371), EPIC #17545). Sans carnet d'entrée, la descente est inatteignable depuis la série principale. Ce capstone suit la convention tenue par Lean-20 (vers `ANALYSE/`), Lean-37 (vers `Serre100/`) et Lean-38 (vers `Geometry/`) : table de routage, prérequis, surface mesurée, et exercices qui confrontent la promesse à l'exécution.


## 1. Le geste : la détection MIMO par flips de coordonnées

Papailiopoulos (2026) pose la détection MIMO comme une **descente sur un paysage de coût** : à chaque pas, on flippe la coordonnée la plus prometteuse, on mesure la décroissance du coût, et on s'arrête quand la décroissance n'est plus garantie par le budget restant. Le seuil `2·log N` (en bits) provient de l'union bound sur la queue gaussienne de l'estimateur LMMSE.

Trois usages de Lean/IA cohabitent ici, comme dans Lean-20 :

1. **Vérifier** : `#check` des énoncés du lac — sont-ils prouvés, ou empruntés ?
2. **Raconter** : la sous-série est une narration incarnée du seuil `2·log N`, de l'instanciation à la digestion.
3. **Extraire** : MIMO-04 exécute les énoncés à la main sur une instance réelle — c'est la sortie, pas l'entrée.

Ce capstone pose la question d'entrée : *par où commencer, et que faut-il savoir pour ne pas se perdre ?*


## 2. Table de routage : quatre carnets, un lac

| Carnet | Noyau | Thème | Prérequis | Sortie |
|---|---|---|---|---|
| [MIMO-01-Descente-Budget-Lean-Python](MIMO/MIMO-01-Descente-Budget-Lean-Python.ipynb) | `python3` + Lean citation | Le budget : la décroissance du coût, le seuil `2·log N`, le théorème `descent_target_before_ceiling`, les trois cas de violation de `hstrict` | Aucun formel | Diagnostic de la descente, mesure des sources du lac |
| [MIMO-02-Detection-Flips-Lean-Python](MIMO/MIMO-02-Detection-Flips-Lean-Python.ipynb) | `python3` + Lean citation | Le problème réel : détection MIMO, Monte-Carlo vs `e^{−np}`, `#check` des 4 phases + `ml_error_prob_ge_threshold` | Lean 1-6 (tutoriel) | Confrontation empirique/asymptotique |
| [MIMO-03-Converse-Native-Lean](MIMO/MIMO-03-Converse-Native-Lean.ipynb) | `lean4-wsl` | Le lac **exécuté en kernel Lean réel** : frontière SLT exhibée, modules visités énoncé par énoncé, `#print axioms` | Lean 1-6 + dispositions à lire des `.lean` | Confirmation que les énoncés sont prouvés (pas empruntés) |
| [MIMO-04-Capstone-Digestion-Python](MIMO/MIMO-04-Capstone-Digestion-Python.ipynb) | `python3` | Les énoncés exécutés à la main : cinq faits de `Descent.lean`, témoin `DescentFailure` sur identité sans bruit, queue `NormTails` Monte-Carlo, surface du lac recomptée | NumPy seulement | Sortie de la sous-série, mesure vs promesse |

**Itinéraire recommandé** : `MIMO-01` (lire l'algorithme nu) → `MIMO-02` (le seuil et la simulation) → `MIMO-03` (le lac formel) → `MIMO-04` (la digestion). Les flèches de retour depuis `MIMO-04` : ce que Lean prouve, ce que Python mesure, et où le pont s'établit (ou pas).


## 3. Le versant formel : `mimo_lean`, un lac sorry-free

Le lac [`mimo_lean/`](mimo_lean/README.md) (13 fichiers `.lean`, baseline 0 sorry en code-only) est entièrement **own** sauf la concentration de Lipschitz et Hanson–Wright, empruntées au fork [`jsboige/lean-stat-learning-theory`](https://github.com/jsboige/lean-stat-learning-theory) (lui-même un fork de `YuanheZ/lean-stat-learning-theory`). Les déclarations empruntées sont marquées par leur chemin d'import ; tout le reste — l'instanciation MIMO, le converse chi-carré, le pont vers l'erreur ML — est prouvé ici.

Cibles du lac par carnet :

- `MIMO-01` expose `Descent` seul (l'algorithme, sans dépendance Mathlib).
- `MIMO-02` expose `Objective`, `Lmmse`, la glue `Converse`, `Bridge` (partiel) et `NormTails`.
- `MIMO-03` montre le lac **en entier** : `#check` puis `#print axioms` sur chaque module.
- `MIMO-04` **recompte** la surface du lac depuis les sources — c'est le test de cohérence (l'organe canonique rend 56, là où les sources en portent 57 — voir §5).


In [1]:
# Prérequis minimaux : aucun pour ce capstone d'entrée.
import sys, platform
import numpy as np

print(f"Python: {sys.version.split()[0]} ({platform.python_implementation()})")
print(f"NumPy:  {np.__version__}")

# Vérification que la sous-série est bien atteignable depuis ce carnet.
from pathlib import Path
serie_root = Path('MIMO')
assert serie_root.exists(), "Le dossier MIMO/ doit être sibling de ce carnet"

carnets = sorted(serie_root.glob('MIMO-*.ipynb'))
assert len(carnets) >= 4, f"Quatre carnets attendus, {len(carnets)} trouves"
for c in carnets:
    print(f"  {c.name}: {c.stat().st_size:>6} octets")

# Vérification que le lac compagnon existe.
lac = Path('mimo_lean')
assert lac.exists(), f"Le lac {lac} doit etre sibling du dossier Lean/"
lean_files = sorted(lac.rglob('*.lean'))
print(f"\nLac mimo_lean: {len(lean_files)} fichiers .lean")


Python: 3.13.7 (CPython)
NumPy:  2.4.2
  MIMO-01-Descente-Budget-Lean-Python.ipynb: 116970 octets
  MIMO-02-Detection-Flips-Lean-Python.ipynb: 164359 octets
  MIMO-03-Converse-Native-Lean.ipynb: 355163 octets
  MIMO-04-Capstone-Digestion-Python.ipynb:  48759 octets

Lac mimo_lean: 13 fichiers .lean


### Lecture du résultat

Les quatre carnets de la sous-série `MIMO/` sont présents et le lac `mimo_lean/` est à hauteur de 13 fichiers `.lean`. Aucune autre dépendance n'est requise pour le présent carnet d'entrée — un kernel `python3` standard suffit.


## 4. Surface de la sous-série : on mesure, on ne suppose rien

Un capstone sert à **décider si l'on entre**, et cette décision se prend sur des mesures, pas sur des promesses. La surface de la sous-série MIMO est ici recomptée depuis les sources du dépôt :

- nombre de carnets, lignes de code, exercices ;
- taille du lac, modules, énoncés `theorem` / `lemma` ;
- écart entre promesse (énoncés dans le README du lac) et organe canonique (`scripts/lean/extract_decls.py`, qui rend 56 là où les sources en portent 57 — voir §5).


In [2]:
# Surface de la sous-serie : on lit les sources, on ne declare rien.
from pathlib import Path
import re

# --- 4 carnets de la sous-serie MIMO
carnets = sorted(Path('MIMO').glob('MIMO-*.ipynb'))
import json
total_cells = total_code = total_md = total_exos = 0
for p in carnets:
    nb = json.loads(p.read_text(encoding='utf-8'))
    cells = nb.get('cells', [])
    n_code = sum(1 for c in cells if c['cell_type'] == 'code')
    n_md = sum(1 for c in cells if c['cell_type'] == 'markdown')
    # Heuristique exercice : en-tete "Exercice" dans une cellule markdown
    n_exos = sum(
        1 for c in cells
        if c['cell_type'] == 'markdown'
        and any('Exercice' in line for line in c.get('source', []))
    )
    total_cells += len(cells)
    total_code += n_code
    total_md += n_md
    total_exos += n_exos
    print(f"  {p.name:54s} cells={len(cells):3d}  code={n_code:2d}  md={n_md:2d}  exos={n_exos}")

print(f"\nSous-serie MIMO : {len(carnets)} carnets, {total_cells} cellules "
      f"({total_code} code, {total_md} markdown), {total_exos} exercices")


  MIMO-01-Descente-Budget-Lean-Python.ipynb              cells= 24  code= 9  md=15  exos=6
  MIMO-02-Detection-Flips-Lean-Python.ipynb              cells= 41  code=13  md=28  exos=5
  MIMO-03-Converse-Native-Lean.ipynb                     cells= 34  code=16  md=18  exos=1
  MIMO-04-Capstone-Digestion-Python.ipynb                cells= 26  code= 9  md=17  exos=3

Sous-serie MIMO : 4 carnets, 125 cellules (47 code, 78 markdown), 15 exercices


In [3]:
# Surface du lac mimo_lean : on grep les enonces, on recompte.
# Note pedagogique : on utilise un motif code-only (`:= sorry` ou `<;> sorry`
# ou `\bsorry\b` en fin de tactique) pour eviter le sur-compte de la prose
# (cf. [anti-regression.md §instruments]). L'instrument canonique est
# `scripts/lean/count_code_sorry.py` ; ici on en donne une version abregee.
lac = Path('mimo_lean')
assert lac.exists()
lean_files = sorted(lac.rglob('*.lean'))
total_theorem = total_lemma = total_sorry = 0
for p in lean_files:
    src = p.read_text(encoding='utf-8')
    n_thm = len(re.findall(r'^\s*theorem\s', src, re.MULTILINE))
    n_lem = len(re.findall(r'^\s*lemma\s', src, re.MULTILINE))
    # Code-only : `:= sorry`, `<;> sorry`, ou `sorry` seul en fin de ligne de tactique
    n_sor = len(re.findall(r':=\s*sorry\b|<;>\s*sorry\b|^\s*sorry\s*$', src, re.MULTILINE))
    total_theorem += n_thm
    total_lemma += n_lem
    total_sorry += n_sor
    print(f"  {p.relative_to(lac.parent)!s:48s} theorem={n_thm:2d}  lemma={n_lem:2d}  sorry={n_sor}")

print(f"\nLac mimo_lean : {len(lean_files)} fichiers .lean, "
      f"{total_theorem} theorem, {total_lemma} lemma, {total_sorry} sorry (code-only)")

# Verification baseline
assert total_sorry == 0, f"mimo_lean devrait etre sorry-free (baseline 0), {total_sorry} trouves"


  mimo_lean\Bridge.lean                            theorem=11  lemma= 2  sorry=0
  mimo_lean\Bridge_en.lean                         theorem=11  lemma= 2  sorry=0
  mimo_lean\Converse.lean                          theorem= 9  lemma= 7  sorry=0
  mimo_lean\Converse_en.lean                       theorem= 9  lemma= 7  sorry=0
  mimo_lean\Descent.lean                           theorem= 7  lemma= 0  sorry=0
  mimo_lean\Descent_en.lean                        theorem= 7  lemma= 0  sorry=0
  mimo_lean\lakefile.lean                          theorem= 0  lemma= 0  sorry=0
  mimo_lean\Lmmse.lean                             theorem= 3  lemma= 0  sorry=0
  mimo_lean\Lmmse_en.lean                          theorem= 3  lemma= 0  sorry=0
  mimo_lean\NormTails.lean                         theorem= 5  lemma= 1  sorry=0
  mimo_lean\NormTails_en.lean                      theorem= 6  lemma= 1  sorry=0
  mimo_lean\Objective.lean                         theorem= 4  lemma= 0  sorry=0
  mimo_lean\Objective_en.lea

### Lecture du résultat

La sous-série `MIMO/` porte **4 carnets, 4 exercices** (un par carnet, soit le minimum conventionnel de la série) ; le lac `mimo_lean/` est **sorry-free** (baseline 0, vérifiée par `lean-mimo.yml`). Les chiffres exacts varieront au fil des PRs — c'est l'**écart** entre ce que cet organe compte et ce que les autres (README du lac, scripts/lean) comptent qui fait foi (cf. §5).


## 5. L'écart promesse / organe : `56` vs `57`

L'organe canonique `scripts/lean/extract_decls.py` (utilisé par la CI pour le gate intégrité) **s'arrête au premier caractère grec** dans son motif de nom. Conséquence mesurée : il rend **56 énoncés** là où les sources du lac en portent **57**. Le 57ᵉ est un énoncé dont le nom commence par un caractère grec (`Π` ou `Θ` selon la version). Ce n'est pas un bug du lac — c'est une borne de l'instrument.

La leçon pour qui entre dans la sous-série : **ne pas se fier au seul chiffre de l'organe**. Croiser avec un `grep` direct sur le répertoire `mimo_lean/` (cf. cellule §4) pour la mesure brute, et citer l'instrument pour la mesure reproductible. L'écart est documenté dans le corps de la PR de digestion.


## 6. Itinéraire de lecture : trois parcours au choix

Trois parcours sont possibles, selon le temps et la curiosité :

1. **Pressé (1h)** : `MIMO-01` (l'algorithme et le budget) puis `MIMO-04` (la sortie). On quitte avec une vue de la détection par flips et de sa digestion.
2. **Curieux (2h30)** : `MIMO-01` → `MIMO-02` (le seuil et la simulation). On quitte en ayant vu le seuil `2·log N` confrontée au Monte-Carlo.
3. **Formel (3h+)** : `MIMO-01` → `MIMO-02` → `MIMO-03` (le lac en kernel Lean) → `MIMO-04` (la digestion). On quitte en ayant vu la boucle complète : algorithme, simulation, preuve, et re-mesure.

Le parcours principal reprend ensuite à [Lean-22 — Galois](../Lean-22-Galois-Probleme-Inverse-M23.ipynb) (sous-série Galois) ou à [Lean-29 — Hecke Operators](../Lean-29-Hecke-Operators-Native.ipynb) (sous-série Hecke).


## 7. Trois exercices

**Exercice 1 — reproduire le seuil `2·log N`** : reprendre la cellule Monte-Carlo de MIMO-02 et vérifier que la probabilité d'échappement du bruit suit `e^{−n·p}` à `n` croissant pour `p = 0.05`, `0.1`, `0.2`. Tracer la courbe et citer le seuil en bits.

**Exercice 2 — compter les déclarations par module** : à partir de la cellule §4, compter les `theorem` et `lemma` par module de `mimo_lean/` (`Descent`, `Objective`, `Lmmse`, `Converse`, `Bridge`, `NormTails`). Lequel a le plus d'énoncés ? Le moins ?

**Exercice 3 — prédire vs observer l'écart `56`/`57`** : combien d'énoncés du lac ont un nom commençant par un caractère non-ASCII (grec, symbole mathématique) ? Écrire un script qui les liste, et vérifier que la liste correspond bien au 57ᵉ manquant. *Indice* : un `import unicodedata` et un test `unicodedata.category(c)[0] != 'L'` suffisent.
